In [1]:
import sqlite3
import pandas as pd

In [3]:
import os

# Read cleaned CSV using the full path
csv_path = r"C:\Users\rezaf\Project\SalesVision\notebooks\salesvision_cleaned.csv"

df_clean = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df_clean.shape)



Dataset loaded successfully!
Dataset Shape: (10194, 21)


In [4]:

# Create SQLite database in the SalesVision root folder
db_path = r"C:\Users\rezaf\Project\SalesVision\salesvision.db"

conn = sqlite3.connect(db_path)

# Load the dataset into a SQL table
df_clean.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

print("Data successfully loaded into SQLite!")



Data successfully loaded into SQLite!


In [5]:
query = """
SELECT COUNT(*) AS total_records
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)

   total_records
0          10194


In [6]:

import sqlite3
import pandas as pd

db_path = r"C:\Users\rezaf\Project\SalesVision\salesvision.db"

conn = sqlite3.connect(db_path)

# Check available tables
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

print(pd.read_sql_query(query, conn))



    name
0  sales


In [8]:

query = """
SELECT COUNT(*) AS total_records
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)



   total_records
0          10194


In [9]:

query = """
SELECT ROUND(SUM(Sales), 2) AS total_sales
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)


   total_sales
0   2326534.35


In [10]:

query = """
SELECT ROUND(SUM(Profit), 2) AS total_profit
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)



   total_profit
0     292296.81


In [11]:

query = """
SELECT
    COUNT(DISTINCT "Order ID") AS unique_orders,
    COUNT(DISTINCT "Customer ID") AS unique_customers,
    COUNT(DISTINCT "Product ID") AS unique_products
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)



   unique_orders  unique_customers  unique_products
0           5111               804             1862


In [15]:

query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM sales
GROUP BY Category
ORDER BY Total_Sales DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)


          Category  Total_Sales  Total_Profit
0       Technology    839893.28     146543.38
1        Furniture    754747.76      19730.00
2  Office Supplies    731893.31     126023.44


In [16]:

query = """
SELECT
    Category,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM sales
GROUP BY Category
ORDER BY Total_Profit DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)


          Category  Total_Profit
0       Technology     146543.38
1  Office Supplies     126023.44
2        Furniture      19730.00


In [17]:

query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(
        SUM(Profit) * 100.0 / SUM(Sales),
        2
    ) AS Profit_Margin_Percent
FROM sales
GROUP BY Category
ORDER BY Profit_Margin_Percent DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)



          Category  Total_Sales  Total_Profit  Profit_Margin_Percent
0       Technology    839893.28     146543.38                  17.45
1  Office Supplies    731893.31     126023.44                  17.22
2        Furniture    754747.76      19730.00                   2.61


In [18]:

query = """
SELECT
    COUNT(*) AS Loss_Records,
    ROUND(SUM(Profit), 2) AS Total_Loss
FROM sales
WHERE Profit < 0;
"""

result = pd.read_sql_query(query, conn)
print(result)


   Loss_Records  Total_Loss
0          1901  -157038.93


In [19]:

query = """
SELECT
    COUNT(*) AS Loss_Records,
    ROUND(SUM(Profit), 2) AS Total_Loss
FROM sales
WHERE Profit < 0;
"""

result = pd.read_sql_query(query, conn)
print(result)



   Loss_Records  Total_Loss
0          1901  -157038.93


In [20]:

query = """
SELECT
    Category,
    COUNT(*) AS Loss_Records,
    ROUND(SUM(Profit), 2) AS Total_Loss
FROM sales
WHERE Profit < 0
GROUP BY Category
ORDER BY Total_Loss ASC;
"""

result = pd.read_sql_query(query, conn)
print(result)


          Category  Loss_Records  Total_Loss
0        Furniture           732   -61675.18
1  Office Supplies           898   -56783.83
2       Technology           271   -38579.92


In [21]:

query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM sales
GROUP BY Category
HAVING SUM(Profit) < 50000
ORDER BY SUM(Profit) ASC;
"""

result = pd.read_sql_query(query, conn)
print(result)



    Category  Total_Sales  Total_Profit
0  Furniture    754747.76       19730.0


In [22]:

queries = {
    "Overall KPIs": """
        SELECT
            ROUND(SUM(Sales), 2) AS Total_Sales,
            ROUND(SUM(Profit), 2) AS Total_Profit,
            COUNT(DISTINCT "Order ID") AS Unique_Orders
        FROM sales;
    """,

    "Category Performance": """
        SELECT
            Category,
            ROUND(SUM(Sales), 2) AS Total_Sales,
            ROUND(SUM(Profit), 2) AS Total_Profit
        FROM sales
        GROUP BY Category
        ORDER BY SUM(Profit) DESC;
    """,

    "Region Performance": """
        SELECT
            Region,
            ROUND(SUM(Sales), 2) AS Total_Sales,
            ROUND(SUM(Profit), 2) AS Total_Profit
        FROM sales
        GROUP BY Region
        ORDER BY SUM(Sales) DESC;
    """,

    "Loss-Making Sub-Categories": """
        SELECT
            "Sub-Category",
            ROUND(SUM(Profit), 2) AS Total_Profit
        FROM sales
        GROUP BY "Sub-Category"
        HAVING SUM(Profit) < 0
        ORDER BY SUM(Profit);
    """,

    "Segment Performance": """
        SELECT
            Segment,
            ROUND(SUM(Sales), 2) AS Total_Sales,
            ROUND(SUM(Profit), 2) AS Total_Profit
        FROM sales
        GROUP BY Segment
        ORDER BY SUM(Sales) DESC;
    """
}

for title, query in queries.items():
    print("\n==========", title, "==========")
    display(pd.read_sql_query(query, conn))




========== Overall KPIs ==========


,Total_Sales,Total_Profit,Unique_Orders
0,2326534.35,292296.81,5111



========== Category Performance ==========


,Category,Total_Sales,Total_Profit
0,Technology,839893.28,146543.38
1,Office Supplies,731893.31,126023.44
2,Furniture,754747.76,19730.00



========== Region Performance ==========


,Region,Total_Sales,Total_Profit
0,West,739813.61,110798.82
1,East,691828.17,94883.26
2,Central,503170.67,39865.31
3,South,391721.91,46749.43



========== Loss-Making Sub-Categories ==========


,Sub-Category,Total_Profit
0,Tables,-17753.21
1,Bookcases,-3632.07
2,Supplies,-1171.39



========== Segment Performance ==========


,Segment,Total_Sales,Total_Profit
0,Consumer,1170659.79,136371.45
1,Corporate,715806.13,94249.64
2,Home Office,440068.43,61675.73
